In [11]:
# ============================================================================
# CELL 1: IMPORTS & CONFIGURATION
# ============================================================================
import os
import re
import json
import pickle
import emoji
import warnings
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from sentence_transformers import SentenceTransformer
from bertopic import BERTopic
import umap

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1200)
pd.set_option('display.max_colwidth', 100)

# Set random seeds for reproducibility
np.random.seed(42)
torch.manual_seed(42)

# Plotting configuration
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams.update({
'figure.figsize': (12, 6),
'figure.dpi': 120,
'font.size': 11
})

print('✓ Libraries loaded successfully')
print(f'✓ PyTorch device: {"CUDA" if torch.cuda.is_available() else "CPU"}')
print(f'✓ Timestamp: {datetime.now().strftime("%Y-%m-%d %H:%M:%S")}')

✓ Libraries loaded successfully
✓ PyTorch device: CPU
✓ Timestamp: 2026-05-29 03:59:47


In [12]:
# ============================================================================
# CELL 2: PATH CONFIGURATION
# ============================================================================
# Base directories
BASE_DIR = Path('.')
DATA_DIR = BASE_DIR / 'data'
RAW_DIR = DATA_DIR / 'raw'
PROCESSED_DIR = DATA_DIR / 'processed'
OUTPUT_DIR = BASE_DIR / 'outputs'
CHECKPOINT_DIR = BASE_DIR / 'checkpoints'

# Create directories if they don't exist
for d in [RAW_DIR, PROCESSED_DIR, OUTPUT_DIR, CHECKPOINT_DIR,
          OUTPUT_DIR / 'polish' / 'visualizations',
          OUTPUT_DIR / 'romanian' / 'visualizations']:
    d.mkdir(parents=True, exist_ok=True)

print('✓ Directory structure verified')
print(f'✓ Base directory: {BASE_DIR.absolute()}')

✓ Directory structure verified
✓ Base directory: c:\Users\andre\OneDrive\Desktop\2_Disertatie_RIPEMC_D.Flanja\Quantitative_Python\YT_Analysis


In [13]:
# ============================================================================
# CELL 3: MODEL CONFIGURATION
# ============================================================================
# Sentiment models (language-specific)
SENTIMENT_MODELS = {
    'pl': 'eevvgg/bert-polish-sentiment-politics',
    'ro': 'readerbench/ro-sentiment'
}

# Sentiment label mappings (will be confirmed after model loading)
SENTIMENT_LABELS = {
    'pl': ['Negative', 'Neutral', 'Positive'],
    'ro': ['Negative', 'Positive']
}

# Embedding model (multilingual, shared)
EMBEDDING_MODEL = 'sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2'

# BERTopic configuration
BERTOPIC_CONFIG = {
    'n_topics': 8,
    'min_topic_size': 10,
    'calculate_probabilities': True
}

# UMAP configuration
UMAP_CONFIG = {
    'n_neighbors': 15,
    'min_dist': 0.1,
    'n_components': 2,
    'metric': 'cosine',
    'random_state': 42
}

# Batch size (CPU-optimized)
BATCH_SIZE = 8

print('✓ Model configuration loaded')
print(f'  - Polish sentiment: {SENTIMENT_MODELS["pl"]}')
print(f'  - Romanian sentiment: {SENTIMENT_MODELS["ro"]}')
print(f'  - Embedding model: {EMBEDDING_MODEL}')
print(f'  - BERTopic topics: {BERTOPIC_CONFIG["n_topics"]}')
print(f'  - Batch size: {BATCH_SIZE}')

✓ Model configuration loaded
  - Polish sentiment: eevvgg/bert-polish-sentiment-politics
  - Romanian sentiment: readerbench/ro-sentiment
  - Embedding model: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
  - BERTopic topics: 8
  - Batch size: 8


In [14]:
# ============================================================================
# CELL 4: CHECKPOINT UTILITY FUNCTIONS
# ============================================================================
def load_checkpoint(lang: str, stage: str) -> pd.DataFrame:
    """Load data from checkpoint if exists."""
    checkpoint_file = PROCESSED_DIR / f"{stage}_{lang}.pkl"
    if checkpoint_file.exists():
        print(f"✓ Loading checkpoint: {checkpoint_file.name}")
        return pd.read_pickle(checkpoint_file)
    else:
        print(f"⚠ Checkpoint not found: {checkpoint_file.name}")
        return None

def save_checkpoint(df: pd.DataFrame, lang: str, stage: str):
    """Save data to checkpoint."""
    checkpoint_file = PROCESSED_DIR / f"{stage}_{lang}.pkl"
    df.to_pickle(checkpoint_file)
    print(f"✓ Checkpoint saved: {checkpoint_file.name}")

def update_pipeline_status(lang: str, block: int, status: str = 'completed'):
    """Update pipeline status JSON."""
    status_file = CHECKPOINT_DIR / 'pipeline_status.json'
    
    if status_file.exists():
        with open(status_file, 'r') as f:
            pipeline_status = json.load(f)
    else:
        pipeline_status = {'pl': {}, 'ro': {}}
    
    pipeline_status[lang][f'block_{block}'] = {
        'status': status,
        'timestamp': datetime.now().isoformat()
    }
    
    with open(status_file, 'w') as f:
        json.dump(pipeline_status, f, indent=2)
    
    print(f"✓ Pipeline status updated: {lang} - Block {block} - {status}")

def check_checkpoint_exists(lang: str, stage: str) -> bool:
    """Check if checkpoint file exists."""
    checkpoint_file = PROCESSED_DIR / f"{stage}_{lang}.pkl"
    return checkpoint_file.exists()

print('✓ Checkpoint utility functions loaded')

✓ Checkpoint utility functions loaded


In [15]:
# ============================================================================
# CELL 5: TEXT CLEANING FUNCTIONS
# ============================================================================
def clean_social_text(text: str, keep_diacritics: bool = True) -> str:
    """
    Clean social media text by removing URLs, emoji, HTML, mentions.
    Preserves diacritics (important for Polish & Romanian).
    """
    if pd.isna(text) or not isinstance(text, str):
        return ''
    
    t = text.strip()
    
    # Remove URLs
    t = re.sub(r'http\S+|www\.\S+', ' ', t)
    
    # Remove HTML tags
    t = re.sub(r'<[^>]+>', ' ', t)
    
    # Remove mentions (@username)
    t = re.sub(r'@\w+', ' ', t)
    
    # Convert emoji to space (don't demojize - removes meaning)
    t = emoji.replace_emoji(t, replace=' ')
    
    # Normalize whitespace
    t = re.sub(r'\s+', ' ', t).strip()
    
    return t

def remove_empty_texts(df: pd.DataFrame, text_col: str = 'clean_text') -> pd.DataFrame:
    """Remove rows with empty or whitespace-only text."""
    before = len(df)
    df = df[df[text_col].str.len() > 0].copy()
    removed = before - len(df)
    print(f"  Removed {removed:,} empty text rows ({100*removed/max(before,1):.1f}%)")
    return df

def remove_duplicates(df: pd.DataFrame, subset: list = None) -> pd.DataFrame:
    """Remove exact duplicates."""
    if subset is None:
        subset = ['video_id', 'author_name', 'comment_text']
        subset = [c for c in subset if c in df.columns]
    
    before = len(df)
    df = df.drop_duplicates(subset=subset, keep='first').copy()
    removed = before - len(df)
    print(f"  Removed {removed:,} duplicate rows ({100*removed/max(before,1):.1f}%)")
    return df

print('✓ Text cleaning functions loaded')

✓ Text cleaning functions loaded


In [16]:
# ============================================================================
# STOPWORDS - ROMANIAN & POLISH (Function Words Only)
# ============================================================================
# Use for preprocessing and topic modeling

STOPWORDS_RO = [
    # Articles
    'un', 'una', 'unei', 'une', 'unui', 'unii', 'unor',
    'o', 'ul', 'ului', 'la', 'le', 'l',
    
    # Prepositions
    'în', 'din', 'prin', 'cu', 'fără', 'de', 'la', 'pe', 'peste',
    'sub', 'lângă', 'contra', 'despre', 'asupra', 'către', 'spre',
    'după', 'până', 'pentru', 'fără', 'înainte', 'înapoi',
    
    # Conjunctions
    'și', 'si', 'dar', 'sau', 'ori', 'că', 'ca', 'căci', 'decât',
    'dacă', 'deși', 'deci', 'așadar', 'astfel', 'întrucât', 'deoarece',
    'fiindcă', 'să', 'încât', 'când', 'unde', 'cum', 'cine', 'ce', 'care',
    'fie', 'nici', 'atât', 'cât',
    
    # Personal Pronouns
    'eu', 'tu', 'el', 'ea', 'noi', 'voi', 'ei', 'ele',
    'mine', 'tine', 'sine', 'îmi', 'îți', 'își', 'ne', 'vă',
    'mi', 'ti', 'i', 'le', 'm', 't', 's', 'l',
    'mă', 'te', 'se',
    
    # Possessive Pronouns
    'meu', 'mea', 'mei', 'mele',
    'tău', 'ta', 'tăi', 'tale',
    'său', 'sa', 'săi', 'sale',
    'nostru', 'noastră', 'noștri', 'noastre',
    'vostru', 'voastră', 'voștri', 'voastre',
    'lor', 'lui', 'ei',
    
    # Demonstrative Pronouns
    'acest', 'aceasta', 'aceste', 'acești', 'aceștia',
    'acel', 'acea', 'acei', 'acele', 'aceia',
    'ăsta', 'ăsta', 'ăștia', 'ăstea', 'ăla', 'aia', 'ăia', 'alea',
    'același', 'aceeași', 'aceiași',
    
    # Indefinite Pronouns
    'cineva', 'ceva', 'oricine', 'orice', 'oriunde',
    'nimeni', 'nimic', 'nicăieri', 'niciodată',
    'unul', 'una', 'unii', 'unele', 'alții', 'altele',
    'fiecare', 'tot', 'toți', 'toate', 'totul',
    'câțiva', 'câteva', 'niște',
    
    # Auxiliary Verbs - A FI (to be)
    'sunt', 'ești', 'este', 'suntem', 'sunteți', 'sînt', 'sînteți',
    'eram', 'erai', 'era', 'erați', 'erau',
    'fi', 'fii', 'fie', 'fiți', 'fiind', 'fost',
    'ar fi', 'să fie', 'o fi',
    
    # Auxiliary Verbs - A AVEA (to have)
    'am', 'ai', 'are', 'avem', 'aveți', 'au',
    'aveam', 'aveai', 'avea', 'aveați', 'aveau',
    'având', 'avut',
    
    # Modal Verbs (common forms)
    'pot', 'poți', 'poate', 'putem', 'puteți',
    'putea', 'puteau', 'ar putea',
    'vreau', 'vrei', 'vrea', 'vrem', 'vreți', 'vor',
    'voia', 'voiau', 'ar vrea',
    
    # Adverbs - Time
    'acum', 'atunci', 'azi', 'ieri', 'mâine', 'mereu',
    'niciodată', 'adesea', 'deja', 'încă', 'mai', 'prea',
    'tot', 'totdeauna', 'întotdeauna',
    
    # Adverbs - Place
    'aici', 'acolo', 'colo', 'dincolo', 'pretutindeni',
    'nicăieri', 'oriunde', 'undeva', 'sus', 'jos',
    'înainte', 'înapoi', 'înăuntru', 'afară', 'departe', 'aproape',
    
    # Adverbs - Manner/Degree
    'așa', 'asa', 'astfel', 'altfel', 'cumva', 'bine', 'rău',
    'foarte', 'prea', 'destul', 'cam', 'oare', 'parcă', 'poate',
    'sigur', 'desigur', 'probabil', 'posibil', 'exact', 'aproximativ',
    
    # Affirmation/Negation
    'da', 'nu', 'ba', 'nici', 'sigur', 'desigur',
    
    # Interjections/Fillers
    'ah', 'oh', 'eh', 'păi', 'ei', 'vai', 'of', 'na', 'uite', 'iată',
    'hmm', 'hm', 'ăă', 'eee', 'mmm',
    'adică', 'practic', 'de fapt', 'gen', 'chestie', 'lucru', 'treabă',
    
    # Numbers (common)
    'zero', 'unu', 'una', 'doi', 'două', 'trei', 'patru', 'cinci',
    'primul', 'prima', 'ultimul', 'ultima',
    
    # English stopwords (appear in Romanian text)
    'the', 'a', 'an', 'and', 'or', 'but', 'in', 'on', 'at', 'to', 'for',
    'of', 'with', 'by', 'from', 'as', 'is', 'was', 'are', 'were', 'been',
    'be', 'have', 'has', 'had', 'do', 'does', 'did', 'will', 'would',
    'could', 'should', 'may', 'might', 'must', 'can',
    'this', 'that', 'these', 'those', 'i', 'you', 'he', 'she', 'it',
    'we', 'they', 'what', 'which', 'who', 'where', 'when', 'why', 'how',
    'all', 'each', 'every', 'both', 'few', 'more', 'most', 'other', 'some',
    'such', 'no', 'nor', 'not', 'only', 'own', 'same', 'so', 'than',
    'too', 'very', 'just', 'also', 'now', 'here', 'there', 'then',
    
    # Platform/Video terms (appear in comments but not useful for topics)
    'video', 'videoclip', 'comment', 'comments', 'comentariu', 'comentarii',
    'reply', 'răspuns', 'răspunsuri', 'like', 'likes', 'distribuire', 'share',
    'subscribe', 'abonare', 'abonat', 'canal', 'channel', 'youtube',
    'algoritm', 'algorithm', 'mulțumim', 'multumim', 'mersi', 'ms',
    ' salut', 'salut', 'ok', 'okay', 'bun', 'buna', 'bine',
]

STOPWORDS_PL = [
    # Articles don't exist in Polish, but we have other function words
    
    # Prepositions
    'w', 'we', 'z', 'ze', 'do', 'na', 'po', 'pod', 'nad', 'przed',
    'za', 'przy', 'bez', 'o', 'obok', 'około', 'wokół', 'przez',
    'prze', 'między', 'pomiędzy', 'spośród', 'znad', 'spod', 'według',
    'wg', 'wzgl', 'względem', 'dzięki', 'zamiast', 'z powodu', 'na skutek',
    
    # Conjunctions
    'i', 'oraz', 'a', 'ale', 'lecz', 'jednak', 'czy', 'lub', 'albo',
    'ani', 'niż', 'jak', 'że', 'żeby', 'aby', 'by', 'gdy', 'kiedy',
    'jeśli', 'jeżeli', 'bo', 'ponieważ', 'gdyż', 'więc', 'zatem',
    'dlatego', 'toteż', 'aczkolwiek', 'chociaż', 'mimo', 'mimo że',
    'podczas gdy', 'zanim', 'aż', 'dopóki', 'dopóki co', 'skoro',
    
    # Personal Pronouns
    'ja', 'ty', 'on', 'ona', 'ono', 'my', 'wy', 'oni', 'one',
    'mnie', 'mnie', 'tobie', 'ci', 'tobą', 'jemu', 'mu', 'jej', 'nią',
    'nam', 'nas', 'wami', 'wam', 'imi', 'nich', 'nim', 'nimi',
    'go', 'ją', 'mi', 'cię', 'mu', 'jej', 'nam', 'was', 'im',
    'się', 'se', 'sobie', 'sobą',
    
    # Possessive Pronouns
    'mój', 'moja', 'moje', 'moi', 'moje',
    'twój', 'twoja', 'twoje', 'twoi', 'twoje',
    'jego', 'jej', 'ich',
    'nasz', 'nasza', 'nasze', 'nasi', 'nasze',
    'wasz', 'wasza', 'wasze', 'wasi', 'wasze',
    'swój', 'swoja', 'swoje', 'swoi', 'swoje',
    
    # Demonstrative Pronouns
    'ten', 'ta', 'to', 'ci', 'te',
    'tamten', 'tamta', 'tamto', 'tamci', 'tamte',
    'taki', 'taka', 'takie', 'tacy', 'takie',
    'jaki', 'jaka', 'jakie', 'jacy', 'jakie',
    'który', 'która', 'które', 'którzy', 'które',
    'czyj', 'czyja', 'czyje', 'czyi', 'czyje',
    
    # Indefinite Pronouns
    'ktoś', 'coś', 'któryś', 'jakiś', 'jakaś', 'jakieś',
    'nikt', 'nic', 'żaden', 'żadna', 'żadne', 'żadni',
    'wszyscy', 'wszystkie', 'wszystko', 'cały', 'cała', 'całe', 'całi',
    'każdy', 'każda', 'każde', 'każdzi',
    'kilka', 'kilku', 'wiele', 'wielu', 'mało', 'mało',
    'coś', 'kogokolwiek', 'czegokolwiek', 'jakikolwiek',
    
    # Auxiliary Verbs - BYĆ (to be)
    'jest', 'są', 'był', 'była', 'było', 'byli', 'były',
    'będzie', 'będą', 'byłem', 'byłam', 'byłeś', 'byłaś',
    'byliśmy', 'byłyśmy', 'byliście', 'byłyście',
    'będę', 'będziesz', 'będziemy', 'będziecie',
    'być', 'bywam', 'bywasz', 'bywa', 'bywamy', 'bywacie', 'bywają',
    'zostać', 'został', 'została', 'zostało', 'zostali', 'zostały',
    
    # Auxiliary Verbs - MIEĆ (to have)
    'mam', 'masz', 'ma', 'mamy', 'macie', 'mają',
    'miał', 'miała', 'miało', 'mieli', 'miały',
    'mieć', 'miałem', 'miałam', 'miałeś', 'miałaś',
    'mieliśmy', 'miałyśmy', 'mieliście', 'miałyście',
    
    # Modal Verbs
    'mogę', 'możesz', 'może', 'możemy', 'możecie', 'mogą',
    'mógł', 'mogła', 'mogło', 'mogli', 'mogły',
    'muszę', 'musisz', 'musi', 'musimy', 'musicie', 'muszą',
    'musiał', 'musiała', 'musiało', 'musieli', 'musiały',
    'chcę', 'chcesz', 'chce', 'chcemy', 'chcecie', 'chcą',
    'chciał', 'chciała', 'chciało', 'chcieli', 'chciały',
    'powinien', 'powinna', 'powinno', 'powinni', 'powinny',
    'warto', 'trzeba', 'należy', 'można',
    
    # Adverbs - Time
    'teraz', 'wtedy', 'dziś', 'dzisiaj', 'wczoraj', 'jutro',
    'zawsze', 'nigdy', 'często', 'rzadko', 'znowu', 'znów',
    'jeszcze', 'już', 'dopiero', 'wciąż', 'nadal', 'dalej',
    'później', 'wcześniej', 'najpierw', 'ostatecznie', 'wreszcie',
    'niebawem', 'wkrótce', 'długo', 'krótko', 'dawno',
    
    # Adverbs - Place
    'tu', 'tutaj', 'tam', 'gdzie', 'gdzieś', 'nigdzie', 'wszędzie',
    'blisko', 'daleko', 'wysoko', 'nisko', 'głęboko', 'szeroko',
    'dołu', 'góry', 'przodu', 'tyłu', 'środka', 'boku',
    'wewnątrz', 'na zewnątrz', 'obok', 'koło', 'przy',
    
    # Adverbs - Manner/Degree
    'tak', 'nie', 'bardzo', 'dosyć', 'dość', 'zbyt', 'za',
    'prawie', 'całkowicie', 'zupełnie', 'pełni', 'częściowo',
    'może', 'chyba', 'pewnie', 'na pewno', 'oczywiście',
    'rzeczywiście', 'faktycznie', 'podobno', 'jakby', 'jak gdyby',
    'dobrze', 'źle', 'lepiej', 'gorzej', 'najlepiej', 'najgorzej',
    'szybko', 'wolno', 'łatwo', 'trudno', 'ciężko', 'prosto',
    'specjalnie', 'celowo', 'przypadkiem', 'losowo',
    
    # Affirmation/Negation
    'tak', 'nie', 'no', 'no tak', 'no nie', 'oj', 'och',
    'pewnie', 'oczywiście', 'z pewnością', 'bez wątpienia',
    
    # Interjections/Fillers
    'ah', 'oh', 'eh', 'oj', 'och', 'hej', 'cześć', 'witaj',
    'no', 'no cóż', 'wiesz', 'wiecie', 'słuchaj', 'słuchajcie',
    'znaczy', 'czyli', 'także', 'również', 'też',
    'jakby', 'jak gdyby', 'tak jakby', 'mniej więcej', 'około',
    
    # Numbers (common)
    'zero', 'jeden', 'jedna', 'jedno', 'dwa', 'dwie', 'trzy',
    'cztery', 'pięć', 'sześć', 'siedem', 'osiem', 'dziewięć', 'dziesięć',
    'pierwszy', 'pierwsza', 'pierwsze', 'ostatni', 'ostatnia', 'ostatnie',
    
    # English stopwords (appear in Polish text too)
    'the', 'a', 'an', 'and', 'or', 'but', 'in', 'on', 'at', 'to', 'for',
    'of', 'with', 'by', 'from', 'as', 'is', 'was', 'are', 'were', 'been',
    'be', 'have', 'has', 'had', 'do', 'does', 'did', 'will', 'would',
    'could', 'should', 'may', 'might', 'must', 'can',
    'this', 'that', 'these', 'those', 'i', 'you', 'he', 'she', 'it',
    'we', 'they', 'what', 'which', 'who', 'where', 'when', 'why', 'how',
    'all', 'each', 'every', 'both', 'few', 'more', 'most', 'other', 'some',
    'such', 'no', 'nor', 'not', 'only', 'own', 'same', 'so', 'than',
    'too', 'very', 'just', 'also', 'now', 'here', 'there', 'then',
    
    # Platform/Video terms (appear in comments but not useful for topics)
    'video', 'film', 'komentarz', 'komentarze', 'comment', 'comments',
    'odpowiedź', 'odpowiedzi', 'reply', 'like', 'lubię', 'polubienie',
    'udostępnij', 'share', 'distribuire', 'subskrybuj', 'subskrypcja',
    'kanał', 'channel', 'youtube', 'algorytm', 'algorithm',
    'dziękuję', 'dzieki', 'thx', 'thanks', 'thank you',
    'cześć', 'hej', 'ok', 'okay', 'dobrze', 'super', 'fajnie',
]

# Convert to sets for faster lookup
STOPWORDS_RO_SET = set(w.lower() for w in STOPWORDS_RO)
STOPWORDS_PL_SET = set(w.lower() for w in STOPWORDS_PL)

print(f'✓ Romanian stopwords: {len(STOPWORDS_RO_SET)} words')
print(f'✓ Polish stopwords: {len(STOPWORDS_PL_SET)} words')


✓ Romanian stopwords: 385 words
✓ Polish stopwords: 511 words


In [17]:
# ============================================================================
# CELL 6: VISUALIZATION UTILITY FUNCTIONS
# ============================================================================
def save_figure(fig, filename: str, lang: str, subdir: str = ''):
    """Save figure to appropriate output directory."""
    if subdir:
        output_path = OUTPUT_DIR / lang / subdir / filename
    else:
        output_path = OUTPUT_DIR / lang / filename
    
    output_path.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(output_path, dpi=150, bbox_inches='tight')
    print(f"✓ Saved: {output_path}")
    plt.close(fig)

def plot_sentiment_bar(df: pd.DataFrame, lang: str, sentiment_col: str = 'sentiment'):
    """Create sentiment distribution bar chart."""
    sentiment_counts = df[sentiment_col].value_counts()
    sentiment_pct = 100 * sentiment_counts / len(df)
    
    fig, ax = plt.subplots(figsize=(10, 6))
    colors = ['#ff6b6b', '#feca57', '#48dbfb']
    
    bars = ax.bar(sentiment_pct.index, sentiment_pct.values, 
                  color=colors[:len(sentiment_pct)], edgecolor='black', alpha=0.8)
    
    ax.set_xlabel('Sentiment', fontsize=12)
    ax.set_ylabel('Percentage (%)', fontsize=12)
    ax.set_title(f'Sentiment Distribution - {lang.upper()} (n={len(df):,})', fontsize=14, fontweight='bold')
    ax.set_ylim(0, max(sentiment_pct.values) * 1.2)
    
    # Add value labels
    for bar, val in zip(bars, sentiment_pct.values):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1, 
                f'{val:.1f}%', ha='center', va='bottom', fontsize=10)
    
    plt.xticks(rotation=45)
    plt.grid(axis='y', alpha=0.3)
    plt.tight_layout()
    
    save_figure(fig, f'{lang}_sentiment_bar.png', lang, 'visualizations')
    return fig

print('✓ Visualization utility functions loaded')

def predict_sentiment_batch(texts: list, model, tokenizer, batch_size: int = BATCH_SIZE) -> list:
    """Predict sentiment for a list of texts in batches."""
    all_predictions = []
    
    for i in tqdm(range(0, len(texts), batch_size), desc='Predicting sentiment'):
        batch_texts = texts[i:i+batch_size]
        
        # Tokenize
        inputs = tokenizer(batch_texts, padding=True, truncation=True, 
                          max_length=128, return_tensors='pt')
        
        # Predict
        with torch.no_grad():
            outputs = model(**inputs)
            predictions = torch.softmax(outputs.logits, dim=1)
            predicted_classes = torch.argmax(predictions, dim=1)
        
        all_predictions.extend(predicted_classes.cpu().numpy())
    
    return all_predictions

print('✓ Sentiment prediction function loaded')



✓ Visualization utility functions loaded
✓ Sentiment prediction function loaded


In [18]:
# ============================================================================
# CELL 7: FINAL STATUS
# ============================================================================
print('='*70)
print('✓ SETUP AND CONFIGURATION COMPLETE')
print('='*70)
print(f'\nReady for analysis. Next steps:')
print(f'  1. Run Polish pipeline: 01_data_loading_pl.ipynb → 06_engagement_metrics_pl.ipynb')
print(f'  2. Run Romanian pipeline: 01_data_loading_ro.ipynb → 06_engagement_metrics_ro.ipynb')
print(f'  3. (Optional) Run comparative analysis: 08_comparative_analysis.ipynb')
print(f'\nCheckpoint directory: {PROCESSED_DIR.absolute()}')
print(f'Output directory: {OUTPUT_DIR.absolute()}')
print('='*70)

✓ SETUP AND CONFIGURATION COMPLETE

Ready for analysis. Next steps:
  1. Run Polish pipeline: 01_data_loading_pl.ipynb → 06_engagement_metrics_pl.ipynb
  2. Run Romanian pipeline: 01_data_loading_ro.ipynb → 06_engagement_metrics_ro.ipynb
  3. (Optional) Run comparative analysis: 08_comparative_analysis.ipynb

Checkpoint directory: c:\Users\andre\OneDrive\Desktop\2_Disertatie_RIPEMC_D.Flanja\Quantitative_Python\YT_Analysis\data\processed
Output directory: c:\Users\andre\OneDrive\Desktop\2_Disertatie_RIPEMC_D.Flanja\Quantitative_Python\YT_Analysis\outputs
